## Active Case Linking GOLD Tables

### Pre-requisites

In [0]:
import dlt

from pyspark.sql import Window
from pyspark.sql.functions import array, col, collect_list, current_timestamp, date_format, expr, lit, regexp_replace, row_number, struct

In [0]:
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()

print(f"env_code: {lz_key}")  # This won't be redacted
print(f"env_name: {env_name}")  # This won't be redacted

KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
print(f"KeyVault_name: {KeyVault_name}") 

In [0]:
# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")

# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
xcutting_storage = f"ingest{lz_key}xcutting{env_name}"

# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for xcutting storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{xcutting_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{xcutting_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{xcutting_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{xcutting_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{xcutting_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
state = "CASE_LINKING"

silver_path = f"abfss://silver@{curated_storage}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
gold_path = f"abfss://gold@{curated_storage}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state}"

aria_ccd_mappings_data_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/all_active_states/ack_audit"

### GOLD - ARIA-CCD Mappings

In [0]:
@dlt.table(
    name="aria_ccd_case_mappings",
    comment="DLT table storing the ARIA CaseNo's to their respective CCD Reference from submission.",
    path=f"{gold_path}/aria_ccd_case_mappings"
)
def aria_ccd_case_mappings():
    aria_ccd_mappings_df = spark.read.format("delta").load(aria_ccd_mappings_data_path)

    df_final = (
        aria_ccd_mappings_df
            .filter("STATUS = 'SUCCESS'")
            .select("CaseNo", "CCDCaseID")
            .withColumn(
                "CaseNo",
                regexp_replace(regexp_replace("CaseNo", r"^APPEALS_|\.json$", ""), "_", "/")
            )
    )

    return df_final

### GOLD - Case Link Combinations

In [0]:
@dlt.table(
    name="case_link_combinations",
    comment="DLT table storing the Case Link combinations.",
    path=f"{gold_path}/case_link_combinations"
)
def case_link_combinations():
    try:
        aria_ccd_mappings_df = dlt.read("aria_ccd_case_mappings")
        case_link_reason_mappings_df = dlt.read("silver_case_link_reason_mappings")
        case_link_groups_df = dlt.read("silver_case_link_groups")
    except:
        aria_ccd_mappings_df = spark.table("hive_metastore.ariadm_active_appeals.aria_ccd_case_mappings")
        case_link_reason_mappings_df = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_reason_mappings")
        case_link_groups_df = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_groups")

    case_link_no_window = Window.partitionBy("LinkNo").orderBy("BirthDate", "CaseNo")

    sorted_case_link_groups = case_link_groups_df.withColumn("row_number", row_number().over(case_link_no_window))

    case_link_combinations = (
        sorted_case_link_groups.alias("from")
            .join(
                sorted_case_link_groups.alias("to"),
                    on=expr("from.LinkNo = to.LinkNo AND from.row_number < to.row_number"),
                    how="inner"
            )
            .join(case_link_reason_mappings_df.alias("clrm"), on=expr("from.ReasonLinkId = clrm.ReasonLinkId"), how="left")
            .select(
                col("from.CaseNo").alias("CaseNoFrom"),
                col("to.CaseNo").alias("CaseNoTo"),
                col("from.ReasonLinkId"),
                col("clrm.key"),
                col("clrm.description")
            )
    )

    df_final = (
        case_link_combinations.alias("clc")
            .join(aria_ccd_mappings_df.alias("from"), on=expr("clc.CaseNoFrom = from.CaseNo"), how="left")
            .join(aria_ccd_mappings_df.alias("to"), on=expr("clc.CaseNoTo = to.CaseNo"), how="left")
            .select(
                col("clc.CaseNoFrom"),
                col("clc.CaseNoTo"),
                col("clc.ReasonLinkId"),
                col("clc.key"),
                col("clc.description"),
                col("from.CCDCaseID").alias("CCDCaseReferenceNumberFrom"),
                col("to.CCDCaseId").alias("CCDCaseReferenceNumberTo")
            )
    )

    return df_final

### GOLD - Case Link Payloads

In [0]:
@dlt.table(
    name="case_link_payloads",
    comment="DLT table storing the Case Link payloads for each CCD reference.",
    path=f"{gold_path}/case_link_payloads"
)
def case_link_payloads():
    try:
        case_link_combinations = dlt.read("case_link_combinations")
    except:
        case_link_combinations = spark.table("hive_metastore.ariadm_active_appeals.case_link_combinations")

    case_link_combinations_to_payload = (
        case_link_combinations
            .filter("CCDCaseReferenceNumberFrom IS NOT NULL AND CCDCaseReferenceNumberTo IS NOT NULL")
            .withColumn(
                "payload",
                struct(
                    col("CCDCaseReferenceNumberTo").alias("id"),
                    struct(
                        lit("Asylum").alias("CaseType"),
                        col("CCDCaseReferenceNumberTo").alias("CaseReference"),
                        array(
                            struct(
                                expr("uuid()").alias("id"),
                                struct(
                                    col("key").alias("Reason"),
                                    col("description").alias("OtherDescription")
                                ).alias("value")
                            )
                        ).alias("ReasonForLink"),
                        date_format(current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss.SSS").alias("CreatedDateTime")
                    ).alias("value")
                )
            )
            .select(
                "CCDCaseReferenceNumberFrom", "payload"
            )
    )

    df_final = (
        case_link_combinations_to_payload
            .groupBy("CCDCaseReferenceNumberFrom")
            .agg(collect_list("payload").alias("caseLinks"))
    )

    return df_final